In [11]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import random
import math

In [12]:
@dataclass
class State:
    R_asset_t: int
    p_t: float
    p_bar_t: float   # preço suavizado
    t: int

In [13]:
def sample_W_t1(rng: random.Random, sigma: float) -> float:
    """
    Gera W_{t+1} = p_hat_{t+1} ~ N(0, sigma^2)
    """
    p_hat_t1 = rng.gauss(0.0, sigma)
    return p_hat_t1


In [14]:
def C(S_t: State, x_t: int) -> float:
    return S_t.p_t * x_t

In [15]:
def S_M(S_t: State, x_t: int, W_t1: float, alpha: float) -> State:
    """
    Implementa:
      R_asset_{t+1} = R_asset_t - x_t
      p_{t+1} = p_t + p_hat_{t+1}
      p_bar_{t+1} = (1-alpha) p_bar_t + alpha p_{t+1}
    """
    R_asset_t1 = max(0, S_t.R_asset_t - x_t)
    p_t1 = S_t.p_t + W_t1
    p_bar_t1 = (1.0 - alpha) * S_t.p_bar_t + alpha * p_t1

    return State(
        R_asset_t=R_asset_t1,
        p_t=p_t1,
        p_bar_t=p_bar_t1,
        t=S_t.t + 1
    )


In [16]:
def X_sell_low(S_t: State, T: int, theta_low: float) -> int:
    """
    X^{sell-low}(S_t | theta_low)
    """
    if S_t.R_asset_t == 0:
        return 0
    if S_t.p_t < theta_low:
        return 1
    if S_t.t == T - 1:
        return 1
    return 0


def X_high_low(S_t: State, T: int, theta_low: float, theta_high: float) -> int:
    """
    X^{high-low}(S_t | theta_low, theta_high)
    """
    if S_t.R_asset_t == 0:
        return 0
    if S_t.p_t < theta_low or S_t.p_t > theta_high:
        return 1
    if S_t.t == T - 1:
        return 1
    return 0


def X_track(S_t: State, T: int, theta_track: float) -> int:
    """
    X^{track}(S_t | theta_track)
    Vende se p_t >= p_bar_t + theta_track
    """
    if S_t.R_asset_t == 0:
        return 0
    if S_t.p_t >= S_t.p_bar_t + theta_track:
        return 1
    if S_t.t == T - 1:
        return 1
    return 0

In [17]:
def simulate_sample_path(
    policy_name: str,
    policy_params: Dict[str, float],
    p_0: float = 100.0,
    T: int = 20,
    sigma: float = 5.0,
    alpha: float = 0.2,
    seed: Optional[int] = None
) -> Tuple[float, List[State], List[int]]:
    rng = random.Random(seed)

    S_t = State(
        R_asset_t=1,
        p_t=p_0,
        p_bar_t=p_0,
        t=0
    )

    states = [S_t]
    decisions: List[int] = []
    F_hat_pi = 0.0

    for _ in range(T):
        if policy_name == "sell_low":
            x_t = X_sell_low(S_t, T, policy_params["theta_low"])
        elif policy_name == "high_low":
            x_t = X_high_low(
                S_t, T,
                policy_params["theta_low"],
                policy_params["theta_high"]
            )
        elif policy_name == "track":
            x_t = X_track(S_t, T, policy_params["theta_track"])
        else:
            raise ValueError(f"Política desconhecida: {policy_name}")

        # restrição x_t <= R_asset_t
        x_t = min(x_t, S_t.R_asset_t)

        F_hat_pi += C(S_t, x_t)
        decisions.append(x_t)

        if x_t == 1:
            break

        W_t1 = sample_W_t1(rng, sigma)
        S_t = S_M(S_t, x_t, W_t1, alpha)
        states.append(S_t)

    return F_hat_pi, states, decisions

In [18]:
def evaluate_policy(
    policy_name: str,
    policy_params: Dict[str, float],
    N: int = 1000,
    p_0: float = 100.0,
    T: int = 20,
    sigma: float = 5.0,
    alpha: float = 0.2,
    base_seed: int = 123
) -> Tuple[float, float]:
    values: List[float] = []

    for n in range(N):
        F_hat_pi, _, _ = simulate_sample_path(
            policy_name=policy_name,
            policy_params=policy_params,
            p_0=p_0,
            T=T,
            sigma=sigma,
            alpha=alpha,
            seed=base_seed + n
        )
        values.append(F_hat_pi)

    F_pi = sum(values) / N

    if N > 1:
        var_hat = sum((v - F_pi) ** 2 for v in values) / (N - 1)
        std_hat = math.sqrt(var_hat)
    else:
        std_hat = 0.0

    return F_pi, std_hat

In [19]:
def search_sell_low(theta_low_grid: List[float], **kwargs) -> Tuple[Dict[str, float], float]:
    best_theta = None
    best_value = float("-inf")

    for theta_low in theta_low_grid:
        F_pi, _ = evaluate_policy(
            policy_name="sell_low",
            policy_params={"theta_low": theta_low},
            **kwargs
        )
        if F_pi > best_value:
            best_value = F_pi
            best_theta = {"theta_low": theta_low}

    return best_theta, best_value


def search_high_low(
    theta_low_grid: List[float],
    theta_high_grid: List[float],
    **kwargs
) -> Tuple[Dict[str, float], float]:
    best_theta = None
    best_value = float("-inf")

    for theta_low in theta_low_grid:
        for theta_high in theta_high_grid:
            if theta_high <= theta_low:
                continue

            F_pi, _ = evaluate_policy(
                policy_name="high_low",
                policy_params={
                    "theta_low": theta_low,
                    "theta_high": theta_high
                },
                **kwargs
            )

            if F_pi > best_value:
                best_value = F_pi
                best_theta = {
                    "theta_low": theta_low,
                    "theta_high": theta_high
                }

    return best_theta, best_value


def search_track(theta_track_grid: List[float], **kwargs) -> Tuple[Dict[str, float], float]:
    best_theta = None
    best_value = float("-inf")

    for theta_track in theta_track_grid:
        F_pi, _ = evaluate_policy(
            policy_name="track",
            policy_params={"theta_track": theta_track},
            **kwargs
        )
        if F_pi > best_value:
            best_value = F_pi
            best_theta = {"theta_track": theta_track}

    return best_theta, best_value

In [20]:
if __name__ == "__main__":
    common_args = {
        "N": 1000,
        "p_0": 100.0,
        "T": 20,
        "sigma": 5.0,
        "alpha": 0.2,
        "base_seed": 42
    }


    F_pi, std_hat = evaluate_policy(
        policy_name="sell_low",
        policy_params={"theta_low": 95.0},
        **common_args
    )
    print("Sell-low")
    print(f"F^pi(S_0) ≈ {F_pi:.4f}")
    print(f"desvio padrão ≈ {std_hat:.4f}")


    F_pi, std_hat = evaluate_policy(
        policy_name="high_low",
        policy_params={"theta_low": 95.0, "theta_high": 110.0},
        **common_args
    )
    print("High-low")
    print(f"F^pi(S_0) ≈ {F_pi:.4f}")
    print(f"desvio padrão ≈ {std_hat:.4f}")


    F_pi, std_hat = evaluate_policy(
        policy_name="track",
        policy_params={"theta_track": 3.0},
        **common_args
    )
    print("Track")
    print(f"F^pi(S_0) ≈ {F_pi:.4f}")
    print(f"desvio padrão ≈ {std_hat:.4f}")


    best_theta, best_value = search_sell_low(
        theta_low_grid=[90, 92, 94, 96, 98, 100],
        **common_args
    )
    print("\nMelhor sell-low:")
    print(best_theta, best_value)

Política sell-low
F^pi(S_0) ≈ 99.9283
desvio padrão ≈ 14.2512

Política high-low
F^pi(S_0) ≈ 100.3602
desvio padrão ≈ 10.4178

Política track
F^pi(S_0) ≈ 100.3280
desvio padrão ≈ 12.2641

Melhor sell-low:
{'theta_low': 94} 100.12418635996484
